# 02 — Video Captioning with Cosmos-Reason2

**What this notebook demonstrates:**
- Loading `CosmosVisionModel` (multimodal: video + image + text)
- Captioning a video using inline `<video>` tags
- Understanding how frames are sampled (fps parameter)

**Key concept:** Cosmos-Reason2 is built on Qwen3-VL architecture. Videos are
sampled at a configurable FPS (default 4), frames are tokenized into vision
tokens, and the VLM generates text descriptions.

---

In [ ]:
import os
os.environ['QWEN_VL_VIDEO_READER'] = 'torchcodec'

import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Check Sample Video

We use `sample.mp4` from the project root. Let's inspect it first using the `video_probe` tool.

In [ ]:
project_root = os.path.abspath('..')
sample_video = os.path.join(project_root, 'sample.mp4')
print(f"📹 Video: {sample_video}")
print(f"   Exists: {os.path.exists(sample_video)}")
print(f"   Size: {os.path.getsize(sample_video) / 1e6:.1f} MB")

In [ ]:
# Use the video_probe tool to get metadata
from strands_cosmos import video_probe

probe_result = video_probe(video_path=sample_video)
print(probe_result['content'][0]['text'][:500])

## Load CosmosVisionModel

The vision model extends the text model with video and image understanding.

**Key parameters:**
- `fps=4` — sample 4 frames per second from video
- `min_vision_tokens=256` — minimum visual tokens per frame
- `max_vision_tokens=8192` — cap to prevent OOM on long videos

In [ ]:
from strands import Agent
from strands_cosmos import CosmosVisionModel

t0 = time.time()
model = CosmosVisionModel(
    model_id="nvidia/Cosmos-Reason2-2B",
    fps=4,
    params={"max_tokens": 1024, "temperature": 0.6},
)
print(f"✅ Vision model loaded in {time.time() - t0:.1f}s")

In [ ]:
agent = Agent(model=model)
print("✅ Agent ready for video understanding")

## Caption the Video

Use the `<video>path</video>` tag to pass video to the model. The VisionModel
extracts frames, processes them through the visual encoder, and generates a caption.

In [ ]:
t0 = time.time()
result = agent(f"Caption in detail: <video>{sample_video}</video>")
print(f"\n⏱  Generation time: {time.time() - t0:.1f}s")

## Try a Specific Question About the Video

In [ ]:
t0 = time.time()
result = agent(f"<video>{sample_video}</video> How many distinct scenes or shot changes are in this video?")
print(f"\n⏱  Generation time: {time.time() - t0:.1f}s")

---
## Summary

| Metric | Value |
|--------|-------|
| Model | Cosmos-Reason2-2B (Vision) |
| Mode | Video → Text |
| FPS sampling | 4 fps |
| Typical latency | 2-5s (depends on video length) |

**Next:** [03_driving_analysis.ipynb](03_driving_analysis.ipynb) — chain-of-thought for driving safety.